# PiT-Mem (PVLDB): 双时态金融 Agent 记忆存储与索引引擎 — Colab Pro 一键复现主控台

本 Notebook 对应论文 **PiT-Mem: A Point-in-Time Bi-Temporal Storage and Indexing Engine for Look-Ahead-Free Agentic Memory** 的完整单卡 A100 实验流水线（`Exp 1 ~ Exp 6D`）。

### 核心架构与实验覆盖：
1. **本地 NVMe 冷热分层与持久化评测 (`Exp 6A–6D`)**：`1,032 B/rec` 计数核心索引数组（`1.92 GiB @ 2M`）+ `3,072 B/rec` 原始 FP32 向量 POSIX `mmap` 按需分页卸载、清单级原子 CSR 快照与 IEEE 802.3 CRC-32 WAL、`PhaseFairSharedMutex` 12 线程混合并发压测，以及 `PostgreSQL 16 + pgvector v0.8.0` (`shared_buffers = 12GB`) 均运行在 `/content/local_ssd/` 本地 NVMe 上。
2. **431,217 条公开去重金融种子语料 + GPU 编码**：基于 ALFRED 宏观多版本流、SEC EDGAR `10-K/10-Q/8-K Item 4.02` 重述流与 394,529 条去重金融披露文本，扩展至 200 万条 `BAAI/bge-base-en-v1.5` (768d) 双时态向量。
3. **端到端系统与智能体实验 (`Exp 1 ~ Exp 6D`)**：涵盖多量级双工况吞吐/召回（`Exp 1`）、跨选择度路由切换（`Exp 2`）、重述元数据吞吐与校准 Token 代价模型及 3 跳突发实测（`Exp 3/3B`）、`Qwen/Qwen2.5-7B-Instruct` 双协议前视泄漏与审计 F1（`Exp 4`）、`RAP-Compact` 槽位保真与跨热冷层统一检索（`Exp 5/5C`）以及流式宏段偏斜消除与并发持久化（`Exp 6A–6D`）。

In [ ]:
# Step 1: 挂载 Google Drive 并将项目高速同步至本地 NVMe (/content/local_ssd)
from google.colab import drive
import os, shutil

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/PiT-Mem_VLDB_Project'
LOCAL_ROOT = '/content/local_ssd/pit-mem'
os.makedirs('/content/local_ssd', exist_ok=True)
if os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
shutil.copytree(DRIVE_ROOT, LOCAL_ROOT, ignore=shutil.ignore_patterns('*.so', 'build'))
%cd /content/local_ssd/pit-mem
!nvidia-smi
!lscpu | head -n 18 && free -h

In [ ]:
# Step 2: 安装 PostgreSQL + pgvector v0.8.0 与依赖，并编译验证 pit_mem_core
!sudo apt-get update -qq && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq build-essential cmake pybind11-dev postgresql postgresql-contrib postgresql-server-dev-all zstd texlive-latex-extra texlive-fonts-extra
!sudo apt-get install -y -qq postgresql-16-pgvector || sudo apt-get install -y -qq postgresql-14-pgvector || true
!pip install -q --upgrade pybind11 "psycopg[binary]" sentence-transformers transformers accelerate pyarrow pandas matplotlib scipy
!pip install --no-build-isolation ./cpp
!PYTHONPATH=. python3 -m pit_mem.benchmarks.test_mw1_mw4_fixes
import pit_mem_core as pm
print('Compiled & verified pit_mem_core successfully! INF_TICK =', pm.INF_TICK)

In [ ]:
# Step 3: 运行 200 万规模归档基准实验（Exp 1 ~ Exp 6D）
!PYTHONUNBUFFERED=1 PYTHONPATH=. python3 -m pit_mem.benchmarks.run_all_experiments \
    --mode colab_pro \
    --work-dir /content/local_ssd/work_2m \
    --raw-seed-dir data/raw_seed \
    --out-dir results/colab_pro \
    --dim 768 \
    --llm-model Qwen/Qwen2.5-7B-Instruct

In [ ]:
# Step 4: 生成 PVLDB 出版级矢量 PDF 图表、LaTeX 指标宏/表格与论文 PDF，并同步回 Google Drive
!PYTHONPATH=. python3 -m pit_mem.plotting.generate_figures results/colab_pro paper/figures
!PYTHONPATH=. python3 -m pit_mem.plotting.generate_latex_macros --results-dir results/colab_pro --out-tex-dir paper
!cd paper && make

import shutil
shutil.copytree('results/colab_pro', f'{DRIVE_ROOT}/results/colab_pro', dirs_exist_ok=True)
shutil.copytree('paper/figures', f'{DRIVE_ROOT}/paper/figures', dirs_exist_ok=True)
shutil.copy2('paper/generated_metrics.tex', f'{DRIVE_ROOT}/paper/generated_metrics.tex')
shutil.copy2('paper/generated_tables.tex', f'{DRIVE_ROOT}/paper/generated_tables.tex')
shutil.copy2('paper/PiT-Mem_PVLDB.pdf', f'{DRIVE_ROOT}/paper/PiT-Mem_PVLDB.pdf')
print('All Colab Pro results, vector figures & PDF synced back to Google Drive:', DRIVE_ROOT)